In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("recruitment_data.csv")
data

,Age,Gender,EducationLevel,ExperienceYears,PreviousCompanies,DistanceFromCompany,InterviewScore,SkillScore,PersonalityScore,RecruitmentStrategy,HiringDecision
0,26,1,2,0,3,26.783828,48,78,91,1,1
1,39,1,4,12,3,25.862694,35,68,80,2,1
2,48,0,2,3,2,9.920805,20,67,13,2,0
3,34,1,2,5,2,6.407751,36,27,70,3,0
4,30,0,1,6,1,43.105343,23,52,85,2,0
...,...,...,...,...,...,...,...,...,...,...,...
1495,48,0,2,3,4,9.183783,66,3,80,3,1
1496,27,1,2,10,3,14.847731,43,97,7,2,0
1497,24,1,1,1,2,4.289911,31,91,58,1,1
1498,48,0,2,4,4,36.299263,9,37,44,2,1


In [3]:
data.isnull().sum()

Age                    0
Gender                 0
EducationLevel         0
ExperienceYears        0
PreviousCompanies      0
DistanceFromCompany    0
InterviewScore         0
SkillScore             0
PersonalityScore       0
RecruitmentStrategy    0
HiringDecision         0
dtype: int64

In [4]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=['HiringDecision'])
y = data.HiringDecision

train_X, test_X, train_y, test_y = train_test_split(X,y,test_size=0.2,random_state=17)

In [5]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=500)
model.fit(train_X,train_y)

LogisticRegression(max_iter=500)

In [26]:
preds = model.predict(test_X)

In [8]:
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

m1 = XGBClassifier()
m2 = CatBoostClassifier()
m3 = LGBMClassifier()

In [9]:
m1.fit(train_X,train_y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [10]:
m2.fit(train_X,train_y)

Learning rate set to 0.011136
0:	learn: 0.6828262	total: 55.4ms	remaining: 55.3s
1:	learn: 0.6720669	total: 57.1ms	remaining: 28.5s
2:	learn: 0.6648460	total: 58.3ms	remaining: 19.4s
3:	learn: 0.6555905	total: 59.9ms	remaining: 14.9s
4:	learn: 0.6432701	total: 61.6ms	remaining: 12.3s
5:	learn: 0.6339425	total: 63.3ms	remaining: 10.5s
6:	learn: 0.6253332	total: 64.8ms	remaining: 9.2s
7:	learn: 0.6163791	total: 66.6ms	remaining: 8.25s
8:	learn: 0.6072493	total: 68.2ms	remaining: 7.51s
9:	learn: 0.5960013	total: 69.9ms	remaining: 6.92s
10:	learn: 0.5889628	total: 71.7ms	remaining: 6.44s
11:	learn: 0.5799751	total: 73.3ms	remaining: 6.04s
12:	learn: 0.5714175	total: 75ms	remaining: 5.69s
13:	learn: 0.5639972	total: 76.5ms	remaining: 5.39s
14:	learn: 0.5572523	total: 78.1ms	remaining: 5.13s
15:	learn: 0.5510907	total: 79.7ms	remaining: 4.9s
16:	learn: 0.5427756	total: 81.3ms	remaining: 4.7s
17:	learn: 0.5355904	total: 83ms	remaining: 4.53s
18:	learn: 0.5305805	total: 84.7ms	remaining: 4.37s

In [11]:
m3.fit(train_X,train_y)

[LightGBM] [Info] Number of positive: 384, number of negative: 816
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003076 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 1200, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.320000 -> initscore=-0.753772
[LightGBM] [Info] Start training from score -0.753772


LGBMClassifier()

In [12]:
p1 = m1.predict(test_X)
p2 = m2.predict(test_X)
p3 = m3.predict(test_X)

In [14]:
from sklearn.ensemble import StackingClassifier

base_estimators = [('XGB',m1),('LGBM',m3),('CatBoost',m2)]
meta_estimator = LogisticRegression()

StackedModel = StackingClassifier(estimators=base_estimators,final_estimator=meta_estimator)

In [15]:
StackedModel.fit(train_X,train_y)

[LightGBM] [Info] Number of positive: 384, number of negative: 816
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000414 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 1200, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.320000 -> initscore=-0.753772
[LightGBM] [Info] Start training from score -0.753772
Learning rate set to 0.011136
0:	learn: 0.6828262	total: 2.69ms	remaining: 2.68s
1:	learn: 0.6720669	total: 4.59ms	remaining: 2.29s
2:	learn: 0.6648460	total: 5.77ms	remaining: 1.92s
3:	learn: 0.6555905	total: 7.58ms	remaining: 1.89s
4:	learn: 0.6432701	total: 9.23ms	remaining: 1.84s
5:	learn: 0.6339425	total: 10.8ms	remaining: 1.8s
6:	learn: 0.6253332	total: 12.5ms	remaining: 1.77s
7:	learn: 0.6163791	total: 14.2ms	remaining: 1.75s
8:	learn: 0.6072493	total: 15.8ms	remaining: 1.73s
9:	learn: 0.5960013	tota

StackingClassifier(estimators=[('XGB',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_...
                                              max_cat_to_onehot=None,
                                              max_delta_step=None,
                                              max_depth=None, max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=None, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=None, ...)),
                               ('LGBM', LGBMClassifier()),
                               ('CatBoost',
                                <catboost.core.CatBoostClassifier object at 0x7e929369c550>)],
                   final_estimator=LogisticRegression())

In [27]:
stackpreds = StackedModel.predict(test_X)

In [28]:
from sklearn.metrics import classification_report

predlist = [preds,p1,p2,p3,stackpreds]
modelnames = ['logreg','xgb','ctb','lgbm','stack']
for i in range(5):
    print(f"Report for {modelnames[i]}")
    print(classification_report(test_y,predlist[i]))

Report for logreg
              precision    recall  f1-score   support

           0       0.89      0.93      0.91       219
           1       0.78      0.70      0.74        81

    accuracy                           0.87       300
   macro avg       0.84      0.82      0.83       300
weighted avg       0.86      0.87      0.86       300

Report for xgb
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       219
           1       0.95      0.86      0.90        81

    accuracy                           0.95       300
   macro avg       0.95      0.92      0.93       300
weighted avg       0.95      0.95      0.95       300

Report for ctb
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       219
           1       0.97      0.89      0.93        81

    accuracy                           0.96       300
   macro avg       0.97      0.94      0.95       300
weighted avg       0.96   

In [29]:
from sklearn.metrics import confusion_matrix
for i in range (5):
    print(f"CM for {modelnames[i]}")
    print(confusion_matrix(test_y,predlist[i]))

CM for logreg
[[203  16]
 [ 24  57]]
CM for xgb
[[215   4]
 [ 11  70]]
CM for ctb
[[217   2]
 [  9  72]]
CM for lgbm
[[215   4]
 [ 11  70]]
CM for stack
[[216   3]
 [  9  72]]


In [19]:
import pickle

In [20]:
with open('ctb_classifier.pkl','wb') as file:
    pickle.dump(m2,file)